# 03 · Agent gateway with Private Service Connect (PSC)

The first two notebooks called agents and MCP servers directly. In an enterprise
deployment you usually put an **agent gateway** in front of them:

* **one authenticated entrypoint** for many backend agents / MCP servers;
* **centralized policy** — token validation, scope checks, rate limits, audit —
  applied at the edge, so backends stay simple and trust the private network;
* **private connectivity** — with **Private Service Connect (PSC)** the gateway
  is reachable over a **private IP inside your VPC**, never the public internet.

The **auth manager** still supplies the caller's credential; the gateway
*validates* it and routes to the right backend. This notebook runs the gateway +
backends locally (localhost stands in for the private PSC endpoint), then shows
the real PSC wiring.


## Architecture

```
   CONSUMER VPC (your project)                    PRODUCER (agents project / Vertex AI)
   ┌───────────────────────────────┐              ┌───────────────────────────────────┐
   │ Agent / caller                │              │            Service attachment       │
   │  AuthManager                  │              │                    ▲                │
   │   • gateway-access (2LO)      │              │        ┌───────────┴───────────┐    │
   │   • gateway-user   (3LO)      │   private    │        │     Agent Gateway      │    │
   │            │ Bearer …         │   traffic    │        │  (validates token,     │    │
   │            ▼                  │   over PSC   │        │   routes by path)      │    │
   │   PSC endpoint (10.0.0.42) ───┼──────────────┼──────▶ │   /agents/*   /mcp/*   │    │
   │   (forwarding rule)           │              │        └────────┬──────┬────────┘    │
   └───────────────────────────────┘              │                 ▼      ▼             │
                                                   │           agent svc   MCP svc       │
                                                   └───────────────────────────────────┘
```

* **Producer** publishes the gateway behind a **service attachment**.
* **Consumer** creates a **PSC endpoint** (a forwarding rule) that gets a private
  VPC IP; the agent dials that IP, and the auth manager provides the bearer the
  gateway checks.


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from auth_provider_demo import AuthManager
from auth_provider_demo.mock_oauth_server import (
    DEMO_CLIENT_ID, DEMO_CLIENT_SECRET, MockOAuthServer,
)

RUN_PRODUCTION = False   # flip to True to exercise the PSC/production cell

idp = MockOAuthServer().start()

manager = AuthManager()
manager.register_two_legged(
    "gateway-access",
    token_url=idp.token_url, client_id=DEMO_CLIENT_ID, client_secret=DEMO_CLIENT_SECRET,
    scope="gateway.invoke",
    description="Service identity for calling agents/MCP through the gateway",
)
manager.register_three_legged(
    "gateway-user",
    token_url=idp.token_url, authorization_url=idp.authorize_url,
    client_id=DEMO_CLIENT_ID, client_secret=DEMO_CLIENT_SECRET,
    redirect_uri="https://app.example.com/oauth/callback",
    scope="gateway.invoke.onbehalf",
    description="On-behalf-of-user authority through the gateway",
)
print("authorizations:", [a["name"] for a in manager.list_authorizations()])

authorizations: ['gateway-access', 'gateway-user']


## Backends behind the gateway (private, no public exposure)

Two tiny services stand in for backends published to the gateway: an **agent**
service and an **MCP** service. They trust the private network (the gateway has
already authenticated the caller), so they carry no auth of their own — exactly
why you *don't* want them publicly reachable.

In [2]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def _make_backend(label, payload_fn):
    class Backend(BaseHTTPRequestHandler):
        def log_message(self, *a): return
        def do_POST(self):
            length = int(self.headers.get("Content-Length", 0))
            body = json.loads(self.rfile.read(length).decode() or "{}")
            # The gateway forwards the authenticated principal in a trusted header.
            principal = self.headers.get("X-Auth-Principal", "unknown")
            out = json.dumps(payload_fn(principal, body)).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(out)))
            self.end_headers()
            self.wfile.write(out)
    srv = ThreadingHTTPServer(("127.0.0.1", 0), Backend)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv, f"http://127.0.0.1:{srv.server_address[1]}"

agent_srv, AGENT_URL = _make_backend(
    "agent", lambda who, body: {"backend": "analytics-agent",
                                "result": "3 critical findings", "as": who})
mcp_srv, MCP_URL = _make_backend(
    "mcp", lambda who, body: {"backend": "drive-mcp",
                              "documents": [f"{who}/plan.doc"], "as": who})
print("agent backend:", AGENT_URL)
print("mcp   backend:", MCP_URL)

agent backend: http://127.0.0.1:35747
mcp   backend: http://127.0.0.1:38613


## The agent gateway — authenticate at the edge, then route

The gateway is the **only** authenticated entrypoint. It:

1. validates the `Authorization: Bearer` token (real gateways verify the JWT
   signature / call token introspection; here we check the token shape);
2. derives the principal and forwards it to the backend in a trusted header;
3. routes by path prefix (`/agents/*`, `/mcp/*`).

In production this gateway is reachable **only** through the PSC endpoint.

In [3]:
import urllib.request, urllib.error

ROUTES = {"/agents/analytics": None, "/mcp/drive": None}  # filled below
ROUTES["/agents/analytics"] = AGENT_URL
ROUTES["/mcp/drive"] = MCP_URL

def _principal_for(token: str):
    if token.startswith("app-access-"):  return "service:gateway-client"
    if token.startswith("user-access-"): return "user:alice"
    return None

class GatewayHandler(BaseHTTPRequestHandler):
    def log_message(self, *a): return
    def _send(self, code, payload, extra=None):
        body = json.dumps(payload).encode()
        self.send_response(code)
        for k, v in (extra or {}).items(): self.send_header(k, v)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_POST(self):
        backend = ROUTES.get(self.path)
        if backend is None:
            return self._send(404, {"error": "no_such_route", "path": self.path})
        auth = self.headers.get("Authorization", "")
        token = auth[7:] if auth.startswith("Bearer ") else ""
        principal = _principal_for(token)
        if principal is None:
            return self._send(401, {"error": "invalid_token"},
                              extra={"WWW-Authenticate": 'Bearer realm="agent-gateway"'})
        # Edge auth passed — forward to the private backend with the principal.
        length = int(self.headers.get("Content-Length", 0))
        payload = self.rfile.read(length)
        req = urllib.request.Request(backend, method="POST", data=payload,
                                     headers={"Content-Type": "application/json",
                                              "X-Auth-Principal": principal})
        with urllib.request.urlopen(req) as r:
            self._send(200, json.loads(r.read().decode()))

gw_srv = ThreadingHTTPServer(("127.0.0.1", 0), GatewayHandler)
# In production this host:port is the PSC endpoint's private VPC IP.
PSC_ENDPOINT = f"http://127.0.0.1:{gw_srv.server_address[1]}"
threading.Thread(target=gw_srv.serve_forever, daemon=True).start()
print("agent gateway (PSC endpoint stand-in):", PSC_ENDPOINT)

agent gateway (PSC endpoint stand-in): http://127.0.0.1:40143


In [4]:
def call_gateway(path, headers=None, message="hello"):
    req = urllib.request.Request(
        f"{PSC_ENDPOINT}{path}", method="POST",
        data=json.dumps({"message": message}).encode(),
        headers={"Content-Type": "application/json", **(headers or {})})
    try:
        with urllib.request.urlopen(req) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode())

### 1. Unauthenticated call through the gateway is rejected

Even on the private endpoint, the gateway enforces auth — **401** without a
valid token.

In [5]:
print(call_gateway("/agents/analytics"))

(401, {'error': 'invalid_token'})


### 2. Reach an agent through the gateway with a **2-legged** token

In [6]:
status, body = call_gateway("/agents/analytics",
                            headers=manager.authorization_headers("gateway-access"))
print("status:", status)
print("routed to:", body["backend"], "| principal:", body["as"], "| result:", body["result"])

status: 200
routed to: analytics-agent | principal: service:gateway-client | result: 3 critical findings


### 3. Reach an MCP server through the same gateway (on behalf of a user)

The user consents once; the manager issues a per-user token; the gateway routes
`/mcp/drive` to the MCP backend, forwarding the user principal.

In [7]:
def simulate_user_consent(consent_url):
    class _NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **k): return None
    opener = urllib.request.build_opener(_NoRedirect)
    try:
        opener.open(consent_url); raise AssertionError("expected redirect")
    except urllib.error.HTTPError as e:
        return e.headers["Location"]

user = "alice"
if manager.needs_consent("gateway-user", user_id=user):
    manager.complete_consent("gateway-user", simulate_user_consent(
        manager.consent_url("gateway-user", user_id=user)))

status, body = call_gateway("/mcp/drive",
                            headers=manager.authorization_headers("gateway-user", user_id=user))
print("status:", status)
print("routed to:", body["backend"], "| principal:", body["as"], "| docs:", body["documents"])

status: 200
routed to: drive-mcp | principal: user:alice | docs: ['user:alice/plan.doc']


One gateway, one auth manager, two backends (agent + MCP), two flows
(service + user) — all behind a single authenticated, privately-reachable
entrypoint.


## Production mapping — Private Service Connect

### Producer side (agents / gateway project): publish the gateway

Expose the gateway's internal load balancer through a **service attachment**:

```bash
# NAT subnet PSC uses for the published service
gcloud compute networks subnets create psc-nat-subnet \
  --network=agents-vpc --region=us-central1 \
  --range=10.10.0.0/24 --purpose=PRIVATE_SERVICE_CONNECT

# Publish the gateway's forwarding rule (internal L4/L7 LB) as a service
gcloud compute service-attachments create agent-gateway-sa \
  --region=us-central1 \
  --producer-forwarding-rule=agent-gateway-fr \
  --connection-preference=ACCEPT_MANUAL \
  --nat-subnets=psc-nat-subnet \
  --consumer-accept-list=CONSUMER_PROJECT_ID=10
```

### Consumer side (your app project): create the PSC endpoint

```bash
# Reserve a private IP for the endpoint in your VPC
gcloud compute addresses create agent-gw-psc-ip \
  --region=us-central1 --subnet=app-subnet --addresses=10.0.0.42

# Create the PSC endpoint (forwarding rule) targeting the service attachment
gcloud compute forwarding-rules create agent-gw-psc \
  --region=us-central1 --network=app-vpc \
  --address=agent-gw-psc-ip \
  --target-service-attachment=projects/AGENTS_PROJECT/regions/us-central1/serviceAttachments/agent-gateway-sa
```

Your agent now dials the gateway at the **private IP `10.0.0.42`** — the
`PSC_ENDPOINT` above is the local stand-in for that address. No traffic touches
the public internet.

### Vertex AI Agent Engine + PSC

Vertex AI supports private connectivity too: deploy Agent Engine / online
prediction with a **PSC interface**, or reach `*-aiplatform.googleapis.com`
through **Private Service Connect for Google APIs**, so calls from the gateway to
the model/agent stay on Google's private network.

### Where the auth manager fits

The gateway validates the same bearer the auth manager issued. Register the
gateway credential once (notebook 00) as `gateway-access` (2-legged) or
`gateway-user` (3-legged); the gateway checks the token (JWT signature / token
introspection) and enforces scopes before routing. **In Gemini Enterprise**,
this is the platform's Authorization attached to the agent, delivered to the
gateway over the private PSC path.


In [8]:
if RUN_PRODUCTION:
    # In production PSC_ENDPOINT would be the reserved private IP, e.g.:
    #   PSC_ENDPOINT = "http://10.0.0.42"
    # and the token would come from your Gemini Enterprise authorization.
    headers = manager.authorization_headers("gateway-access")
    print("Would call PSC endpoint with headers:", {k: v[:24] + "..." for k, v in headers.items()})
else:
    print("RUN_PRODUCTION is False — using the local gateway stand-in above.")

RUN_PRODUCTION is False — using the local gateway stand-in above.


## Summary

* An **agent gateway** gives you one authenticated entrypoint, centralized
  policy, and (with **PSC**) private-IP connectivity for your agents + MCP
  servers.
* The **auth manager** issues the credential; the **gateway validates** it at the
  edge and routes to private backends — the same 2-legged / 3-legged flows from
  notebooks 01–02, now behind a private endpoint.
* PSC wiring is a **producer service attachment** + **consumer endpoint**; the
  code that calls the endpoint is unchanged — only the address becomes private.

This completes the series: auth manager (00) → A2A (01) → MCP (02) → gateway +
PSC (03).


In [9]:
for s in (gw_srv, agent_srv, mcp_srv):
    s.shutdown(); s.server_close()
idp.stop()
print("servers stopped.")

servers stopped.
